IMPORT LIBRARY 

In [1]:
import string
import pandas as pd
import re
import nltk
import contractions
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet, words
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from collections import Counter

Download NLTK data 

In [2]:
# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('omw-1.4')
nltk.download('words')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\amirb\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\amirb\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\amirb\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\amirb\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\amirb\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\amirb\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!


True

 Load the dataset


In [3]:
# Load dataset
df = pd.read_csv('translated_ExportedComment3.csv')

# Display total number of rows
print("Dataset loaded successfully.")
print("Total rows:", len(df))


Dataset loaded successfully.
Total rows: 10338


# Remove duplicate entries

In [4]:
# Show number of rows before removing duplicates and handling missing values
print("Before cleaning:")
print("Total rows:", len(df))
print("Missing Translated_Comment:", df['Translated_Comment'].isnull().sum())
print("Duplicate Translated_Comment:", df.duplicated(subset=['Translated_Comment']).sum())

# Remove duplicates and handle missing values
df = df.drop_duplicates(subset=['Translated_Comment'])
df['Translated_Comment'] = df['Translated_Comment'].fillna('').astype(str)

# Show number of rows after cleaning
print("\nAfter cleaning:")
print("Total rows:", len(df))
print("Missing Translated_Comment:", df['Translated_Comment'].isnull().sum())
print("Duplicate Translated_Comment:", df.duplicated(subset=['Translated_Comment']).sum())


Before cleaning:
Total rows: 10338
Missing Translated_Comment: 107
Duplicate Translated_Comment: 440

After cleaning:
Total rows: 9898
Missing Translated_Comment: 0
Duplicate Translated_Comment: 0


# Handle Missing or Non-String Values

In [5]:
# Display total rows before removing rows that start with links
print("Before removing rows starting with links:")
print("Total rows:", len(df))

# Function to check if text starts with a link
def starts_with_link(text):
    return bool(re.match(r'^\s*(http://|https://|www\.)', text))

# Remove rows starting with links
df = df[~df['Translated_Comment'].apply(starts_with_link)]

# Handle missing or non-string values
df['Translated_Comment'] = df['Translated_Comment'].fillna('').astype(str)

# Display total rows after removal
print("\nAfter removing rows starting with links:")
print("Total rows:", len(df))

Before removing rows starting with links:
Total rows: 9898

After removing rows starting with links:
Total rows: 9738


# Remove URLs, hashtags, and emojis

In [6]:
# Display total rows before text cleaning
print("Before text cleaning:")
print("Total rows:", len(df))
print("Sample comment before cleaning:")
print(df['Translated_Comment'].iloc[0])

# Updated emoji removal function with more comprehensive patterns
def remove_emojis(text):
    # Extended emoji pattern covering more ranges
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002500-\U00002BEF"  # chinese chars (might catch some symbols)
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001f926-\U0001f937"
        u"\U00010000-\U0010ffff"
        u"\u2640-\u2642" 
        u"\u2600-\u2B55"
        u"\u200d"
        u"\u23cf"
        u"\u23e9"
        u"\u231a"
        u"\ufe0f"  # dingbats
        u"\u3030"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

# Updated clean_text function
def clean_text(text):
    # First remove emojis
    text = remove_emojis(text)
    
    # Then apply other cleaning steps
    text = text.lower()
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub(r"\b\d+\b", "", text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('[’"”…]', '', text)
    
    # Contraction handling
    text = contractions.fix(text)
    
    # Additional emoji pass (in case any slipped through)
    text = remove_emojis(text)
    # removing short form:

    text = re.sub("x", 'not', text)
    text = re.sub("isn't", 'is not', text)
    text = re.sub("he's", 'he is', text)
    text = re.sub("wasn't", 'was not', text)
    text = re.sub("there's", 'there is', text)
    text = re.sub("couldn't", 'could not', text)
    text = re.sub("won't", 'will not', text)
    text = re.sub("they're", 'they are', text)
    text = re.sub("she's", 'she is', text)
    text = re.sub("There's", 'there is', text)
    text = re.sub("wouldn't", 'would not', text)
    text = re.sub("haven't", 'have not', text)
    text = re.sub("That's", 'That is', text)
    text = re.sub("you've", 'you have', text)
    text = re.sub("He's", 'He is', text)
    text = re.sub("what's", 'what is', text)
    text = re.sub("weren't", 'were not', text)
    text = re.sub("we're", 'we are', text)
    text = re.sub("hasn't", 'has not', text)
    text = re.sub("you'd", 'you would', text)
    text = re.sub("shouldn't", 'should not', text)
    text = re.sub("let's", 'let us', text)
    text = re.sub("they've", 'they have', text)
    text = re.sub("You'll", 'You will', text)
    text = re.sub("i'm", 'i am', text)
    text = re.sub("we've", 'we have', text)
    text = re.sub("it's", 'it is', text)
    text = re.sub("don't", 'do not', text)
    text = re.sub("that´s", 'that is', text)
    text = re.sub("I´m", 'I am', text)
    text = re.sub("it’s", 'it is', text)
    text = re.sub("she´s", 'she is', text)
    text = re.sub("he’s'", 'he is', text)
    text = re.sub('I’m', 'I am', text)
    text = re.sub('I’d', 'I did', text)
    text = re.sub("he’s'", 'he is', text)
    text = re.sub('there’s', 'there is', text)

    return text

df['Cleaned_Comment'] = df['Translated_Comment'].apply(clean_text)


# Display total rows after text cleaning
print("\nAfter text cleaning:")
print("Total rows:", len(df))
print("Sample comment after cleaning:")
print(df['Cleaned_Comment'].iloc[0])  # optional: show the cleaned example

Before text cleaning:
Total rows: 9738
Sample comment before cleaning:
Arrogant .. also pegkhinat ni .. mistakenly roll his mat.


<>:38: SyntaxWarning: invalid escape sequence '\S'
<>:38: SyntaxWarning: invalid escape sequence '\S'
C:\Users\amirb\AppData\Local\Temp\ipykernel_20104\351342022.py:38: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub('https?://\S+|www\.\S+', '', text)



After text cleaning:
Total rows: 9738
Sample comment after cleaning:
arrogant  also pegkhinat ni  mistakenly roll his mat


# Change To Lower Case

In [7]:
# Convert text to lowercase (Normalization)
df['Normalized_Comment'] = df['Cleaned_Comment'].str.lower()

# Tokenization with TF-H and TF-1 removal

# Initialize stopwords
stop_words = set(stopwords.words('english'))

# Tokenization with contraction handling and stopword removal
def tokenize_and_remove_stopwords(text):
    expanded_text = contractions.fix(text)  # Expand contractions
    tokens = word_tokenize(expanded_text)
    return [word for word in tokens if word.lower() not in stop_words]

df['Tokenized_Comment'] = df['Normalized_Comment'].apply(tokenize_and_remove_stopwords)

In [8]:
# Initialize stopwords
stop_words = set(stopwords.words('english'))

# Define negation words to keep (add more if needed)
negation_words = {'not', 'no', 'nor', 'never', 'none', 'neither', 'nobody', 
                  'nothing', 'nowhere', 'hardly', 'scarcely', 'barely'}

# Define protected political terms (case-insensitive)
protected_terms = {
    'pm', 'bn', 'umno', 'party', 'vote', 'votes', 'gta', 'candidate', 'dap',
    'election', 'pkr', 'government', 'minister', 'support', 'majority', 'opposition',
    'polls', 'voter', 'prime', 'politics', 'parties', 'political', 'mandate', 'leaders',
    'candidates', 'ge15', 'winning', 'campaign', 'najib', 'anwar', 'gombak', 'tun',
    'zahid', 'yassin', 'loss', 'win'
}
sentiment_words = {
    'corrupt', 'hate', 'love', 'support', 'good', 'bad', 'liar', 'strong',
    'weak', 'clean', 'cheat', 'honest', 'fail', 'win', 'loss', 'problem',
    'help', 'respect', 'stupid', 'smart', 'happy', 'angry'
}

# Combine all protected terms (political + sentiment + negation)
protected_terms = protected_terms.union(sentiment_words).union(negation_words)

# Modified stopword removal that preserves protected AND negation terms
def tokenize_and_remove_stopwords(text):
    # Expand contractions first
    expanded_text = contractions.fix(text)
    tokens = word_tokenize(expanded_text)
    
    return [
        word 
        for word in tokens 
        if (word.lower() not in stop_words) 
        or (word.lower() in protected_terms)  # Keep if in protected/negation list
    ]

# Apply tokenization and stopword removal
df['Tokenized_Comment'] = df['Normalized_Comment'].apply(tokenize_and_remove_stopwords)

# Calculate word frequencies
all_tokens = [token for sublist in df['Tokenized_Comment'] for token in sublist]
word_freq = Counter(all_tokens)
total_words_before = len(all_tokens)  # Total word count before any removal

# Get TF-H (excluding protected terms from consideration)
top_n = 20
tf_h_with_freq = word_freq.most_common(top_n)

print("\n=== TF-H Analysis ===")
print(f"Top {top_n} most frequent words:")
for i, (word, freq) in enumerate(tf_h_with_freq, 1):
    print(f"{i}. {word}: {freq} occurrences")

# Extract just the words for reference
tf_h = [word for word, freq in tf_h_with_freq]

# Get TF-1 (words that appear only once, excluding protected terms)
tf_1 = [
    word 
    for word, freq in word_freq.items() 
    if freq == 1 and word.lower() not in protected_terms
]
print("\n=== TF-1 Analysis ===")
print(f"Number of hapax legomena (words appearing once): {len(tf_1)}")
print("\nSample of TF-1 words (first 15000):")
print(tf_1[:15000])

# Vocabulary statistics before any removal
print("\n=== Vocabulary Statistics ===")
print(f"Total words in corpus before any removal: {total_words_before}")
print(f"Unique words before TF-H/TF-1 analysis: {len(word_freq)}")
print(f"TF-H words identified (not removed): {len(tf_h)}")
print(f"TF-1 words to remove: {len(tf_1)}")

# Modified removal function that only removes TF-1 (hapax) words, preserving TF-H
def remove_tf1(tokens):
    return [
        word 
        for word in tokens 
        if (word.lower() in protected_terms) or (word not in tf_1)
    ]

# Apply removal (only TF-1 words removed; TF-H words remain)
df['Tokenized_Comment'] = df['Tokenized_Comment'].apply(remove_tf1)

# Vocabulary statistics after TF-1 removal
remaining_words = [word for sublist in df['Tokenized_Comment'] for word in sublist]
remaining_freq = Counter(remaining_words)
total_words_after = len(remaining_words)  # Total word count after removing TF-1

print("\n=== After TF-1 Removal ===")
print(f"Total words in corpus after removing TF-1: {total_words_after}")
print(f"Words removed (TF-1 only): {total_words_before - total_words_after}")
print(f"Percentage of words kept: {(total_words_after / total_words_before) * 100:.2f}%")
print(f"Unique words remaining: {len(remaining_freq)}")
print(f"Percentage of original vocabulary kept: {len(remaining_freq) / len(word_freq) * 100:.2f}%")

# Verify protected terms preservation
protected_counts = {term: remaining_freq.get(term, 0) for term in protected_terms}
print("\n=== Protected Term Counts ===")
for term, count in sorted(protected_counts.items(), key=lambda x: x[1], reverse=True):
    if count > 0:
        print(f"{term}: {count}")



=== TF-H Analysis ===
Top 20 most frequent words:
1. not: 2104 occurrences
2. pm: 1066 occurrences
3. people: 888 occurrences
4. bn: 812 occurrences
5. umno: 716 occurrences
6. tun: 513 occurrences
7. best: 434 occurrences
8. win: 418 occurrences
9. party: 414 occurrences
10. want: 400 occurrences
11. vote: 360 occurrences
12. la: 344 occurrences
13. ph: 339 occurrences
14. time: 331 occurrences
15. change: 324 occurrences
16. pn: 304 occurrences
17. zahid: 300 occurrences
18. no: 297 occurrences
19. good: 296 occurrences
20. support: 296 occurrences

=== TF-1 Analysis ===
Number of hapax legomena (words appearing once): 5810

Sample of TF-1 words (first 15000):
['pegkhinat', 'mistakenly', '¤²ð', '»', '¹vote', 'zemin', 'altermate', 'hoihoi', 'yahoi', 'slanderous', 'tnet', 'zuraida', 'lamp', 'poles', 'sent', 'arrangance', 'orphanage', 'construction', 'syria', 'palestine', 'gombqk', 'jumps', 'jimin', 'gombakð', 'dodiscover', 'kitol', 'wifeyou', 'drums', 'reformers', 'ancient', 'ltl', 'l

# Lemmatization


In [9]:
# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Convert NLTK POS tags to WordNet POS tags
def get_wordnet_pos(tag):
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag[0].upper(), wordnet.NOUN)

# Lemmatization function
def lemmatize_text(tokens):
    pos_tags = pos_tag(tokens)
    return [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]

df['Lemmatized_Comment'] = df['Tokenized_Comment'].apply(lemmatize_text)
df['Lemmatized_Comment'] = df['Lemmatized_Comment'].apply(lambda x: ' '.join(x))


# Remove special characters


In [10]:
# Function to remove special characters, repeated dots (....), question marks, and exclamation marks
def remove_special_chars(text):
    text = re.sub(r'\.{2,}', '', text)  # Remove repeated dots
    text = re.sub(r'[?!]', '', text)  # Remove ? and !
    text = re.sub(r'[^a-zA-Z0-9.,\' ]+', '', text)  # Remove other special characters
    return text

df['Final_Comment'] = df['Lemmatized_Comment'].apply(remove_special_chars)

print("=== Phase 8: After Removing Special Characters ===")
print(df[['Lemmatized_Comment', 'Final_Comment']])


=== Phase 8: After Removing Special Characters ===
                                      Lemmatized_Comment  \
0                              arrogant also ni roll mat   
1      no space opportunity traitor make sure lose ge...   
2      gombak voter traitor let u teach azmin politic...   
3      may allah make easy d anwar ibrahim good prime...   
4                                         arrogant court   
...                                                  ...   
10333                                result reject bn ph   
10334                                                bee   
10335                                               best   
10336                                                 ok   
10337  matthew sagunting matthew sagunting sweet mout...   

                                           Final_Comment  
0                              arrogant also ni roll mat  
1      no space opportunity traitor make sure lose ge...  
2      gombak voter traitor let u teach azmin polit

# Remove non-standard English words using NLTK

In [11]:
# Load standard English vocabulary
english_vocab = set(words.words())

# Add manually allowed Malaysian political terms
malaysian_terms = {
    "dap", "bn", "pas", "pn", "ph", "umno", "gta",
    "pkr", "warisan", "gps", "grs", "muda"# party full names components
}

# Combine all whitelist terms
whitelist = malaysian_terms


# Function to keep standard words and Malaysian keywords; collect removed ones
def keep_standard_or_malaysian_words(text):
    if not isinstance(text, str):
        return "", []

    tokens = re.findall(r'\b\w+\b', text.lower())  # tokenize and normalize
    kept = [word for word in tokens if word in english_vocab or word in whitelist]
    removed = [word for word in tokens if word not in kept]

    return ' '.join(kept), removed


# Apply function to keep standard English + Malaysian terms
df[['Cleaned_Standard', 'Removed_Words']] = df['Final_Comment'].apply(
    lambda x: pd.Series(keep_standard_or_malaysian_words(x))
)

# Stats
total_removed_words = df['Removed_Words'].apply(len).sum()
rows_before = len(df)

# Remove rows that no longer contain any meaningful standard/political terms
df = df[df['Cleaned_Standard'].str.strip().astype(bool)]
rows_after = len(df)

print(f"\n=== Phase 8.5: Non-Standard Word Filtering with Political Exemptions ===")
print(f"Total removed non-standard words: {total_removed_words}")
print(f"Rows before filtering: {rows_before}")
print(f"Rows after filtering: {rows_after}")

# Sample comparison
print("\n📋 Sample comparison (before vs after):")
print(df[['Final_Comment', 'Cleaned_Standard', 'Removed_Words']].sample(10))

# Finalize column
df['Final_Comment'] = df['Cleaned_Standard']
df.drop(columns=['Cleaned_Standard', 'Removed_Words'], inplace=True)

# Assign Mentioned Party based on keywords in Final_Comment
def extract_parties(text):
    text = text.lower()
    parties = set()

    if 'umno' in text:
        parties.add('UMNO')
    if 'bn' in text:
        parties.add('BN')
    if 'dap' in text:
        parties.add('DAP')
    if 'pkr' in text:
        parties.add('PKR')
    if 'pas' in text:
        parties.add('PAS')
    if 'ph' in text:
        parties.add('PH')
    if 'pn' in text:
        parties.add('PN')
    if 'gta' in text:
        parties.add('GTA')
    if 'warisan' in text:
        parties.add('WARISAN')
    if 'muda' in text:
        parties.add('MUDA')
    if 'bersatu' in text:
        parties.add('BERSATU')
    if 'gps' in text:
        parties.add('GPS')
    if 'grs' in text:
        parties.add('GRS')

    return list(parties) if parties else ['OTHER']


df['Mentioned_Parties'] = df['Final_Comment'].apply(extract_parties)



=== Phase 8.5: Non-Standard Word Filtering with Political Exemptions ===
Total removed non-standard words: 10914
Rows before filtering: 9738
Rows after filtering: 8870

📋 Sample comparison (before vs after):
                                           Final_Comment  \
9528                                 may mrs win big win   
1620              allah almighty even season flood allah   
813       gta not citizen people wise choose snake place   
4027                                  ckp not thief lahh   
6089                                              center   
936                       turn around long find sympathy   
10010  ok no problem not make country money keep peop...   
509                          haaa mcm no spirit applause   
1728                                     week lp say win   
9954   la la la la la la la la la la la la la la la l...   

                                        Cleaned_Standard  \
9528                                     may win big win   
1620      

# Save the preprocessed dataset

In [12]:
# Save the preprocessed dataset
df.to_csv('preprocessed_dataset.csv', index=False)
print("=== Phase 9: Preprocessed Dataset Saved ===")

# Combine both columns for splitting
comment_party_df = df[['Final_Comment', 'Mentioned_Parties']]

# Split full dataset
train_df, test_df = train_test_split(comment_party_df, test_size=0.2, random_state=42)

# Save full train/test datasets
train_df.to_csv('train_dataset.csv', index=False)
test_df.to_csv('test_dataset.csv', index=False)
print("Saved train_dataset.csv and test_dataset.csv for the full dataset.")

=== Phase 9: Preprocessed Dataset Saved ===
Saved train_dataset.csv and test_dataset.csv for the full dataset.


In [13]:
# 1. First 1000 rows
small_1000 = df.head(1000)
small_1000.to_csv('preprocessed_dataset_1000.csv', index=False)
train_1000, test_1000 = train_test_split(small_1000[['Final_Comment','Mentioned_Parties']],
                                         test_size=0.2, random_state=42)
train_1000.to_csv('train_dataset_1000.csv', index=False)
test_1000.to_csv('test_dataset_1000.csv', index=False)
print("Saved preprocessed_dataset_1000.csv and its train/test splits.")

Saved preprocessed_dataset_1000.csv and its train/test splits.


In [14]:
# 2. First 3000 rows
small_3000 = df.head(3000)
small_3000.to_csv('preprocessed_dataset_3000.csv', index=False)
train_3000, test_3000 = train_test_split(small_3000[['Final_Comment','Mentioned_Parties']],
                                         test_size=0.2, random_state=42)
train_3000.to_csv('train_dataset_3000.csv', index=False)
test_3000.to_csv('test_dataset_3000.csv', index=False)
print("Saved preprocessed_dataset_3000.csv and its train/test splits.")

Saved preprocessed_dataset_3000.csv and its train/test splits.


In [15]:
# 3. First 5000 rows
small_5000 = df.head(5000)
small_5000.to_csv('preprocessed_dataset_5000.csv', index=False)
train_5000, test_5000 = train_test_split(small_5000[['Final_Comment','Mentioned_Parties']],
                                         test_size=0.2, random_state=42)
train_5000.to_csv('train_dataset_5000.csv', index=False)
test_5000.to_csv('test_dataset_5000.csv', index=False)
print("Saved preprocessed_dataset_5000.csv and its train/test splits.")

Saved preprocessed_dataset_5000.csv and its train/test splits.
